In [12]:
# %% 셀 1: 모델 + 데이터 로드 (평가용)
import os, json, random, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y
D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
D_FF = 512
DROPOUT = 0.0
INTRA_CHUNK = 512
SPATIAL_CHUNK = 128
BEST_TH = 0.65

CKPT_PATH = "./model/8_layout_vit_patch_mask_focal_smooth/epoch_012.pt"

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    fname = os.path.basename(path)[:-5]
    stt_path = os.path.join(STT_DIR, channel, fname + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])

print(f"✅ eval 영상: {len(eval_samples):,}개")
print(f"   텔롭 있음: {sum(1 for s in eval_samples if s['instances']):,}  없음: {sum(1 for s in eval_samples if not s['instances']):,}")


# ── 모델 정의 ──
class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.slot_proj = nn.Sequential(
            nn.Linear(SLOT_DIM, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.norm = nn.LayerNorm(d_model)

    def forward(self, slot_feats, slot_mask):
        N, K, _ = slot_feats.shape
        device = slot_feats.device
        dtype = slot_feats.dtype
        tokens = self.slot_proj(slot_feats)
        d = tokens.shape[-1]
        has_any = slot_mask.any(dim=1)
        pooled = torch.zeros(N, d, device=device, dtype=dtype)
        if has_any.any():
            vi = has_any.nonzero(as_tuple=True)[0]
            vt = tokens[vi]
            vm = slot_mask[vi]
            vp = ~vm
            V = vt.shape[0]
            count = vm.sum(dim=1, keepdim=True).float()
            count_norm = count / MAX_ACTIVE_PER_FRAME
            mt = vt.masked_fill(vp.unsqueeze(-1), 0.0)
            mean_pool = mt.sum(dim=1) / count.clamp(min=1)
            mfm = vt.masked_fill(vp.unsqueeze(-1), float("-inf"))
            max_pool = mfm.max(dim=1).values
            q = self.pool_query.expand(V, -1, -1)
            a = torch.bmm(q, vt.transpose(1, 2)) / (d ** 0.5)
            a = a.masked_fill(vp.unsqueeze(1), float("-inf"))
            a = F.softmax(a, dim=-1)
            attn_pool = torch.bmm(a, vt).squeeze(1)
            combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
            pooled[vi] = self.norm(self.combine(combined)).to(dtype)
        return pooled


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL, n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.intra_frame = IntraFrameModule(d_model)
        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64), nn.GELU(), nn.Linear(64, d_model))
        self.temporal_norm = nn.LayerNorm(d_model)
        tl = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True, activation="gelu")
        self.temporal_transformer = nn.TransformerEncoder(tl, num_layers=n_layers_temporal,
            enable_nested_tensor=False)
        self.patch_pos_emb = nn.Parameter(torch.randn(1, N_PATCHES, d_model) * 0.02)
        self.spatial_norm = nn.LayerNorm(d_model)
        sl = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True, activation="gelu")
        self.spatial_transformer = nn.TransformerEncoder(sl, num_layers=n_layers_spatial,
            enable_nested_tensor=False)
        self.patch_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, PATCH_SIZE * PATCH_SIZE))

    def forward(self, channel_ids, telop_feats, telop_mask, time_feats, frame_mask):
        B, T, K, _ = telop_feats.shape
        device = telop_feats.device
        dtype = telop_feats.dtype
        BT = B * T
        sf = telop_feats.reshape(BT, K, SLOT_DIM)
        sm = telop_mask.reshape(BT, K)
        fl = []
        for s in range(0, BT, INTRA_CHUNK):
            e = min(s + INTRA_CHUNK, BT)
            fl.append(self.intra_frame(sf[s:e], sm[s:e]))
        ft = torch.cat(fl, dim=0).reshape(B, T, -1)
        ch = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(ft + ch + time)
        to = self.temporal_transformer(tokens, src_key_padding_mask=~frame_mask)
        tf = to.reshape(BT, -1)
        vf = frame_mask.reshape(BT)
        vi = vf.nonzero(as_tuple=True)[0]
        nv = vi.shape[0]
        al = torch.zeros(BT, GRID_H, GRID_W, device=device, dtype=dtype)
        if nv > 0:
            vc = tf[vi]
            queries = vc.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)
            cl = []
            for s in range(0, nv, SPATIAL_CHUNK):
                e = min(s + SPATIAL_CHUNK, nv)
                so = self.spatial_transformer(queries[s:e])
                cl.append(self.patch_head(so))
            ap = torch.cat(cl, dim=0)
            ml = (ap.reshape(-1, N_PATCHES_Y, N_PATCHES_X, PATCH_SIZE, PATCH_SIZE)
                  .permute(0, 1, 3, 2, 4).contiguous().reshape(-1, GRID_H, GRID_W))
            al[vi] = ml.to(al.dtype)
        return al.reshape(B, T, GRID_H, GRID_W)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✅ 모델 로드: {CKPT_PATH}")
print(f"   eval_loss={ckpt['eval_loss']:.4f}  eval_metrics={ckpt['eval_metrics']}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:11<00:00, 5859.87it/s]


✅ eval 영상: 3,320개
   텔롭 있음: 2,984  없음: 336
✅ 모델 로드: ./model/8_layout_vit_patch_mask_focal_smooth/epoch_012.pt
   eval_loss=0.0137  eval_metrics={'P': np.float64(0.35972486112905006), 'R': np.float64(0.8374904158775422), 'F1': np.float64(0.49565119165570704), 'bestF1': np.float64(0.5835099075248027)}


In [13]:
# %% 셀 2: 전체 eval set 추론 + JSON + 영상 생성
import cv2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

plt.rc('font', family='NanumGothic')

FRAME_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/8_telop_position_smoothing_test"
VIS_FPS = 10
VIS_H = 640
VIS_W = 360
GAP = 10
HEADER_H = 50

cmap_gt = matplotlib.colormaps["Blues"]
cmap_pred = matplotlib.colormaps["coolwarm"]

total_w = VIS_W * 3 + GAP * 2
total_h = VIS_H + HEADER_H

try:
    PIL_FONT = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 20)
    PIL_FONT_SM = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 16)
except:
    PIL_FONT = ImageFont.load_default()
    PIL_FONT_SM = ImageFont.load_default()


def get_swap_channel(orig_channel, video_name, candidates):
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    r = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return r.choice(pool)


def prepare_inputs(sample):
    instances = sample["instances"]
    duration = max(sample["duration"], 0.1)
    n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
    n_inst = len(instances)

    times = np.arange(n_frames, dtype=np.float32) / FPS
    time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
    t_norm = times / max(duration, 1.0)
    time_feats[:, 0] = t_norm
    time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
    time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

    if n_inst == 0:
        telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
        telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        gt_masks = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
        active_matrix = np.zeros((n_frames, 0), dtype=np.bool_)
        return telop_feats, telop_mask, time_feats, gt_masks, active_matrix, n_frames

    inst_starts = np.array([inst["start"] for inst in instances])
    inst_ends   = np.array([inst["end"]   for inst in instances])
    inst_tlens  = np.array([inst["text_len"] for inst in instances])
    inst_cx = np.array([inst["x"] for inst in instances])
    inst_cy = np.array([inst["y"] for inst in instances])
    inst_w  = np.array([inst["w"] for inst in instances])
    inst_h  = np.array([inst["h"] for inst in instances])

    channel_emb = text2emb.get(sample["channel"], ZERO_EMB)
    video_emb   = text2emb.get(sample["video_name"], ZERO_EMB)
    inst_embs = torch.stack([text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
    ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
    vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

    stt_sims = np.zeros(n_inst, dtype=np.float32)
    has_stts = np.zeros(n_inst, dtype=np.float32)
    stt_segments = sample["stt_segments"]
    if len(stt_segments) > 0:
        stt_starts = np.array([seg["start"] for seg in stt_segments])
        stt_ends   = np.array([seg["end"]   for seg in stt_segments])
        stt_embs   = torch.stack([text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
        for i in range(n_inst):
            mid = (inst_starts[i] + inst_ends[i]) / 2
            stt_active = (stt_starts <= mid) & (stt_ends > mid)
            stt_active_idx = np.where(stt_active)[0]
            if len(stt_active_idx) > 0:
                stt_sims[i] = F.cosine_similarity(
                    inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                has_stts[i] = 1.0

    active_matrix = (inst_starts[None, :] <= times[:, None] + 0.05) & (inst_ends[None, :] > times[:, None])

    telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
    telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
    for fi in range(n_frames):
        active_idx = np.where(active_matrix[fi])[0]
        if len(active_idx) > 0:
            sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
            sorted_idx = active_idx[sorted_order]
            for sj, ai in enumerate(sorted_idx):
                telop_feats[fi, sj, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                telop_feats[fi, sj, 1] = ch_sims[ai]
                telop_feats[fi, sj, 2] = vid_sims[ai]
                telop_feats[fi, sj, 3] = stt_sims[ai]
                telop_feats[fi, sj, 4] = has_stts[ai]
                telop_mask[fi, sj] = True

    gt_masks = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
    for j in range(n_inst):
        cx, cy = int(inst_cx[j]), int(inst_cy[j])
        w, h   = int(inst_w[j]),  int(inst_h[j])
        x0 = max(0, cx - w // 2)
        y0 = max(0, cy - h // 2)
        x1 = min(GRID_W, x0 + w)
        y1 = min(GRID_H, y0 + h)
        if x1 <= x0 or y1 <= y0:
            continue
        active_frames = np.where(active_matrix[:, j])[0]
        if len(active_frames) > 0:
            gt_masks[active_frames[:, None, None],
                     np.arange(y0, y1)[None, :, None],
                     np.arange(x0, x1)[None, None, :]] = 1.0

    return telop_feats, telop_mask, time_feats, gt_masks, active_matrix, n_frames


def run_inference(telop_feats, telop_mask, time_feats, n_frames, cond_channel_id):
    with torch.no_grad():
        cid = torch.tensor([cond_channel_id], dtype=torch.long, device=device)
        tf_t = torch.from_numpy(telop_feats).unsqueeze(0).to(device)
        tm_t = torch.from_numpy(telop_mask).unsqueeze(0).to(device)
        ti_t = torch.from_numpy(time_feats).unsqueeze(0).to(device)
        fm_t = torch.ones(1, n_frames, dtype=torch.bool, device=device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_logits = model(cid, tf_t, tm_t, ti_t, fm_t)
        return torch.sigmoid(pred_logits[0]).cpu().numpy()


def compute_eval_metrics(gt_masks, pred_prob, active_matrix, threshold=BEST_TH):
    n_frames = gt_masks.shape[0]
    pred_bin = (pred_prob > threshold).astype(np.float32)
    gt_bin = gt_masks.astype(np.bool_)
    pred_bool = pred_bin.astype(np.bool_)

    tp_all = (pred_bool & gt_bin).sum()
    fp_all = (pred_bool & ~gt_bin).sum()
    fn_all = (~pred_bool & gt_bin).sum()
    p_all = tp_all / (tp_all + fp_all + 1e-8)
    r_all = tp_all / (tp_all + fn_all + 1e-8)
    f1_all = 2 * p_all * r_all / (p_all + r_all + 1e-8)

    intersection = (pred_bool & gt_bin).sum()
    union = (pred_bool | gt_bin).sum()
    iou_all = intersection / (union + 1e-8)

    gt_pos = gt_bin.sum()
    pred_pos = pred_bool.sum()
    coverage_ratio = pred_pos / (gt_pos + 1e-8)

    frame_metrics = []
    for fi in range(n_frames):
        gt_f = gt_bin[fi]
        pred_f = pred_bool[fi]
        tp = (pred_f & gt_f).sum()
        fp = (pred_f & ~gt_f).sum()
        fn = (~pred_f & gt_f).sum()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        inter = (pred_f & gt_f).sum()
        uni = (pred_f | gt_f).sum()
        iou = inter / (uni + 1e-8)
        n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
        frame_metrics.append({
            "frame": fi, "t_sec": round(fi / FPS, 1), "n_active": n_active,
            "gt_positive": int(gt_f.sum()), "pred_positive": int(pred_f.sum()),
            "P": round(float(p), 4), "R": round(float(r), 4),
            "F1": round(float(f1), 4), "IoU": round(float(iou), 4),
            "count_MAE": abs(int(pred_f.sum()) - int(gt_f.sum())),
        })

    return {
        "aggregate": {
            "P": round(float(p_all), 4), "R": round(float(r_all), 4),
            "F1": round(float(f1_all), 4), "IoU": round(float(iou_all), 4),
            "coverage_ratio": round(float(coverage_ratio), 4),
            "gt_positive_total": int(gt_pos), "pred_positive_total": int(pred_pos),
            "threshold": threshold,
        },
        "per_frame": frame_metrics,
    }


def find_frame_dir(sample):
    ch_pos_dir = os.path.join(POS_DIR, sample["channel"])
    if not os.path.isdir(ch_pos_dir):
        return ""
    frame_dir = os.path.join(FRAME_DIR, sample["channel"], sample["file_id"])
    if os.path.isdir(frame_dir):
        return frame_dir
    return ""


def write_video(output_path, frame_dir, gt_masks, pred_prob, active_matrix, n_frames, sample, cond_channel):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, VIS_FPS, (total_w, total_h))
    for fi in range(n_frames):
        canvas = np.zeros((total_h, total_w, 3), dtype=np.uint8)
        frame_num = fi + 1
        frame_path = os.path.join(frame_dir, f"frame_{frame_num:08d}.jpg")
        if os.path.exists(frame_path):
            orig = cv2.imread(frame_path)
            orig = cv2.resize(orig, (VIS_W, VIS_H))
        else:
            orig = np.zeros((VIS_H, VIS_W, 3), dtype=np.uint8)
        gt = gt_masks[fi]
        gt_resized = cv2.resize(gt, (VIS_W, VIS_H), interpolation=cv2.INTER_NEAREST)
        gt_panel = (cmap_gt(gt_resized)[:, :, :3] * 255).astype(np.uint8)
        gt_panel = cv2.cvtColor(gt_panel, cv2.COLOR_RGB2BGR)
        pred = pred_prob[fi]
        pred_resized = cv2.resize(pred, (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR)
        pred_panel = (cmap_pred(pred_resized)[:, :, :3] * 255).astype(np.uint8)
        pred_panel = cv2.cvtColor(pred_panel, cv2.COLOR_RGB2BGR)
        x0 = 0
        x1 = VIS_W + GAP
        x2 = VIS_W * 2 + GAP * 2
        canvas[HEADER_H:, x0:x0 + VIS_W] = orig
        canvas[HEADER_H:, x1:x1 + VIS_W] = gt_panel
        canvas[HEADER_H:, x2:x2 + VIS_W] = pred_panel
        t_sec = fi / FPS
        n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
        canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(canvas_rgb)
        draw = ImageDraw.Draw(pil_img)
        draw.text((x0 + 10, 5), "Original", font=PIL_FONT, fill=(255, 255, 255))
        draw.text((x1 + 10, 5), "GT", font=PIL_FONT, fill=(255, 255, 255))
        draw.text((x2 + 10, 5), f"Pred ({cond_channel})", font=PIL_FONT, fill=(255, 255, 255))
        draw.text((10, 30), f"t={t_sec:.1f}s  active={n_active}  [{sample['channel']}]",
                  font=PIL_FONT_SM, fill=(180, 180, 180))
        canvas = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        writer.write(canvas)
    writer.release()


def save_eval_data(json_path, sample, cond_channel, metrics):
    record = {
        "orig_channel": sample["channel"],
        "video_name": sample["video_name"],
        "file_id": sample["file_id"],
        "cond_channel": cond_channel,
        "n_instances": len(sample["instances"]),
        "duration": sample["duration"],
        "n_frames": len(metrics["per_frame"]),
        "model_ckpt": CKPT_PATH,
        "aggregate": metrics["aggregate"],
        "per_frame": metrics["per_frame"],
    }
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)


# ── 메인 루프 ──
for si, sample in enumerate(tqdm(eval_samples, desc="전체 eval")):
    orig_ch = sample["channel"]
    file_id = sample["file_id"]
    orig_ch_id = channel2id[orig_ch]

    swap_ch = get_swap_channel(orig_ch, sample["video_name"], channels)
    swap_ch_id = channel2id[swap_ch]

    telop_feats, telop_mask, time_feats, gt_masks, active_matrix, n_frames = \
        prepare_inputs(sample)

    frame_dir = find_frame_dir(sample)

    # orig
    orig_json = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{orig_ch}.json")
    orig_mp4  = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{orig_ch}.mp4")
    if not os.path.exists(orig_json):
        pred_orig = run_inference(telop_feats, telop_mask, time_feats, n_frames, orig_ch_id)
        metrics_orig = compute_eval_metrics(gt_masks, pred_orig, active_matrix)
        write_video(orig_mp4, frame_dir, gt_masks, pred_orig, active_matrix, n_frames, sample, orig_ch)
        save_eval_data(orig_json, sample, orig_ch, metrics_orig)

    # swap
    swap_json = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{swap_ch}.json")
    swap_mp4  = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{swap_ch}.mp4")
    if not os.path.exists(swap_json):
        pred_swap = run_inference(telop_feats, telop_mask, time_feats, n_frames, swap_ch_id)
        metrics_swap = compute_eval_metrics(gt_masks, pred_swap, active_matrix)
        write_video(swap_mp4, frame_dir, gt_masks, pred_swap, active_matrix, n_frames, sample, swap_ch)
        save_eval_data(swap_json, sample, swap_ch, metrics_swap)

print(f"\n✅ 완료: {len(eval_samples)}개 영상 × 2 (orig+swap) → {OUTPUT_DIR}/")

전체 eval: 100%|██████████| 3320/3320 [8:09:29<00:00,  8.85s/it]   


✅ 완료: 3320개 영상 × 2 (orig+swap) → ./data/8_telop_position_smoothing_test/


In [14]:
# %% 셀 3: JSON 결과 로드 + 채널 통계
import os, json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

OUTPUT_DIR = "./data/8_telop_position_smoothing_test"
POS_DIR = "./data/8_telop_position"

orig_map = {}
swap_map = {}
json_count = 0

for channel in tqdm(sorted(os.listdir(OUTPUT_DIR)), desc="JSON 로드"):
    ch_dir = os.path.join(OUTPUT_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for file_id in os.listdir(ch_dir):
        vid_dir = os.path.join(ch_dir, file_id)
        if not os.path.isdir(vid_dir):
            continue
        for fname in os.listdir(vid_dir):
            if not fname.endswith(".json"):
                continue
            path = os.path.join(vid_dir, fname)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    rec = json.load(f)
            except:
                continue
            key = (rec["orig_channel"], rec.get("file_id", rec.get("video_name", "")))
            if rec["orig_channel"] == rec["cond_channel"]:
                orig_map[key] = rec
            else:
                swap_map[key] = rec
            json_count += 1

print(f"\n✅ JSON 로드: {json_count:,}개")
print(f"   orig: {len(orig_map):,}개  swap: {len(swap_map):,}개")

paired_keys = sorted(set(orig_map.keys()) & set(swap_map.keys()))
print(f"   페어: {len(paired_keys):,}개")

# ── 채널 density bucket ──
channel_stats = {}
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    n_videos = 0
    density_sum = 0.0
    for fname in os.listdir(ch_dir):
        if not fname.endswith(".json"):
            continue
        try:
            with open(os.path.join(ch_dir, fname), "r") as f:
                data = json.load(f)
        except:
            continue
        instances = data.get("instances", [])
        n_videos += 1
        if instances:
            starts = np.array([inst["start_sec"] for inst in instances])
            ends = np.array([inst["end_sec"] for inst in instances])
            durs = np.clip(ends - starts, 0, None)
            vid_len = float(ends.max())
            if vid_len > 0:
                density_sum += float(durs.sum()) / vid_len
    if n_videos > 0:
        channel_stats[channel] = {"n_videos": n_videos, "avg_density": density_sum / n_videos}

stats_df = pd.DataFrame([{"channel": ch, **s} for ch, s in channel_stats.items()])
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
N_BUCKETS = 5
stats_df["bucket"] = pd.qcut(stats_df["avg_density"], q=N_BUCKETS, labels=list(range(N_BUCKETS))).astype(int)
channel2bucket = dict(zip(stats_df["channel"], stats_df["bucket"]))

print(f"\n📊 채널 density bucket 분포")
print(stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"]).to_string())

JSON 로드: 100%|██████████| 664/664 [00:07<00:00, 90.07it/s] 



✅ JSON 로드: 6,640개
   orig: 3,320개  swap: 3,320개
   페어: 3,320개

📊 채널 density bucket 분포
        count       min        max      mean
bucket                                      
0         133  0.011535   0.664213  0.294358
1         133  0.685286   1.688121  1.160411
2         132  1.689540   3.240800  2.519783
3         133  3.257812   4.646314  3.939302
4         133  4.648890  16.820411  6.215769


In [15]:
# %% 셀 4: 절대적 품질 — Pixel-level F1 + IoU
results = []
for key in paired_keys:
    rec = orig_map[key]
    agg = rec["aggregate"]
    results.append({
        "channel": key[0], "file_id": key[1],
        "n_instances": rec["n_instances"],
        "P": agg["P"], "R": agg["R"], "F1": agg["F1"],
        "IoU": agg["IoU"], "coverage_ratio": agg["coverage_ratio"],
        "gt_positive": agg["gt_positive_total"],
        "pred_positive": agg["pred_positive_total"],
    })

print(f"📊 Pixel-level 메트릭 (GT vs pred_orig, n={len(results):,})")
print(f"\n  {'메트릭':<20} {'mean':>8} {'median':>8} {'std':>8} {'min':>8} {'max':>8}")
print(f"  {'─'*20} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for col in ["P", "R", "F1", "IoU", "coverage_ratio"]:
    vals = np.array([r[col] for r in results])
    print(f"  {col:<20} {vals.mean():>8.3f} {np.median(vals):>8.3f} {vals.std():>8.3f} {vals.min():>8.3f} {vals.max():>8.3f}")

f1s = np.array([r["F1"] for r in results])
print(f"\n📊 F1 분포")
print(f"  {'구간':<15} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*15} {'─'*8} {'─'*8}")
for (lo, hi), label in zip(
    [(0,0.1),(0.1,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,0.9),(0.9,1.001)],
    ["0~0.1","0.1~0.2","0.2~0.4","0.4~0.6","0.6~0.8","0.8~0.9","0.9~1.0"]):
    mask = (f1s >= lo) & (f1s < hi)
    print(f"  {label:<15} {mask.sum():>8} {mask.sum()/len(f1s)*100:>7.1f}%")

gt_ns = np.array([r["n_instances"] for r in results])
ious = np.array([r["IoU"] for r in results])

print(f"\n📊 인스턴스 수 구간별")
print(f"  {'구간':<15} {'n':>6} {'F1 mean':>10} {'F1 med':>10} {'IoU mean':>10}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*10}")
for i, (lo, hi) in enumerate([(0,1),(1,10),(10,100),(100,1000),(1000,10000)]):
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0: continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {f1s[mask].mean():>10.3f} {np.median(f1s[mask]):>10.3f} {ious[mask].mean():>10.3f}")

gt_zero = gt_ns == 0
gt_nonzero = gt_ns > 0
print(f"\n📊 GT=0: {gt_zero.sum()}개")
pred_pos_zero = np.array([r["pred_positive"] for r in results])[gt_zero]
print(f"  pred=0: {(pred_pos_zero == 0).sum()} ({(pred_pos_zero == 0).sum()/max(gt_zero.sum(),1)*100:.1f}%)")
print(f"  pred>0: {(pred_pos_zero > 0).sum()} ({(pred_pos_zero > 0).sum()/max(gt_zero.sum(),1)*100:.1f}%) ← false positive")
print(f"\n📊 GT>0: {gt_nonzero.sum()}개")
print(f"  F1 mean: {f1s[gt_nonzero].mean():.3f}, median: {np.median(f1s[gt_nonzero]):.3f}")
print(f"  IoU mean: {ious[gt_nonzero].mean():.3f}, median: {np.median(ious[gt_nonzero]):.3f}")

📊 Pixel-level 메트릭 (GT vs pred_orig, n=3,320)

  메트릭                      mean   median      std      min      max
  ──────────────────── ──────── ──────── ──────── ──────── ────────
  P                       0.412    0.483    0.337    0.000    1.000
  R                       0.409    0.432    0.365    0.000    1.000
  F1                      0.391    0.442    0.338    0.000    1.000
  IoU                     0.302    0.284    0.282    0.000    1.000
  coverage_ratio       7137319277.928    0.875 147150878481.859    0.000 4212600000000.000

📊 F1 분포
  구간                   샘플수       비율
  ─────────────── ──────── ────────
  0~0.1               1223    36.8%
  0.1~0.2              109     3.3%
  0.2~0.4              267     8.0%
  0.4~0.6              428    12.9%
  0.6~0.8              875    26.4%
  0.8~0.9              339    10.2%
  0.9~1.0               79     2.4%

📊 인스턴스 수 구간별
  구간                   n    F1 mean     F1 med   IoU mean
  ─────────────── ────── ────────── ────────── ───

In [16]:
# %% 셀 5: Coverage ratio (과다/과소 예측)
cov_results = [r for r in results if r["n_instances"] > 0]
covs = np.array([r["coverage_ratio"] for r in cov_results])
gt_ns_cov = np.array([r["n_instances"] for r in cov_results])

print(f"📊 Coverage ratio (GT>0만, n={len(cov_results):,})")
print(f"  mean: {covs.mean():.3f}  median: {np.median(covs):.3f}  std: {covs.std():.3f}")

print(f"\n  {'구간':<20} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*20} {'─'*8} {'─'*8}")
for (lo, hi), label in zip(
    [(0,0.5),(0.5,0.8),(0.8,1.2),(1.2,2.0),(2.0,5.0),(5.0,float('inf'))],
    ["< 0.5 (과소)","0.5~0.8","0.8~1.2 (적정)","1.2~2.0","2.0~5.0","≥ 5.0 (과대)"]):
    mask = (covs >= lo) & (covs < hi)
    print(f"  {label:<20} {mask.sum():>8} {mask.sum()/len(covs)*100:>7.1f}%")

print(f"\n📊 인스턴스 수 구간별 coverage")
print(f"  {'구간':<15} {'n':>6} {'mean':>8} {'median':>8}")
print(f"  {'─'*15} {'─'*6} {'─'*8} {'─'*8}")
for i, (lo, hi) in enumerate([(0,1),(1,10),(10,100),(100,1000),(1000,10000)]):
    mask = (gt_ns_cov >= lo) & (gt_ns_cov < hi)
    if mask.sum() == 0: continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {covs[mask].mean():>8.3f} {np.median(covs[mask]):>8.3f}")

📊 Coverage ratio (GT>0만, n=2,984)
  mean: 0.911  median: 0.943  std: 1.688

  구간                        샘플수       비율
  ──────────────────── ──────── ────────
  < 0.5 (과소)                900    30.2%
  0.5~0.8                   289     9.7%
  0.8~1.2 (적정)             1089    36.5%
  1.2~2.0                   631    21.1%
  2.0~5.0                    59     2.0%
  ≥ 5.0 (과대)                 16     0.5%

📊 인스턴스 수 구간별 coverage
  구간                   n     mean   median
  ─────────────── ────── ──────── ────────
      1~10          722    0.825    0.052
     10~100        1735    0.920    0.985
    100~1000        522    1.006    1.046
   1000~10000         5    0.541    0.400


In [17]:
# %% 셀 6: 상대적 차이 — Wilcoxon signed-rank test
from scipy.stats import wilcoxon

rows_d = []
for key in paired_keys:
    orig_agg = orig_map[key]["aggregate"]
    swap_agg = swap_map[key]["aggregate"]
    bucket = channel2bucket.get(key[0], -1)
    rows_d.append({
        "channel": key[0], "file_id": key[1], "bucket": bucket,
        "n_instances": orig_map[key]["n_instances"],
        "orig_F1": orig_agg["F1"], "swap_F1": swap_agg["F1"],
        "orig_IoU": orig_agg["IoU"], "swap_IoU": swap_agg["IoU"],
        "orig_coverage": orig_agg["coverage_ratio"],
        "swap_coverage": swap_agg["coverage_ratio"],
    })

orig_f1  = np.array([r["orig_F1"] for r in rows_d])
swap_f1  = np.array([r["swap_F1"] for r in rows_d])
orig_iou = np.array([r["orig_IoU"] for r in rows_d])
swap_iou = np.array([r["swap_IoU"] for r in rows_d])
orig_cov = np.array([r["orig_coverage"] for r in rows_d])
swap_cov = np.array([r["swap_coverage"] for r in rows_d])

delta_f1  = orig_f1 - swap_f1
delta_iou = orig_iou - swap_iou
delta_cov = np.abs(1 - swap_cov) - np.abs(1 - orig_cov)

gt_ns_d   = np.array([r["n_instances"] for r in rows_d])
buckets_d = np.array([r["bucket"] for r in rows_d])

print(f"📊 상대적 차이 (Δ = orig - swap, n={len(rows_d):,})")
print(f"\n  {'메트릭':<20} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")
for name, delta in [("F1", delta_f1), ("IoU", delta_iou), ("Coverage (|1-x|)", delta_cov)]:
    pos_pct = (delta > 0).sum() / len(delta) * 100
    nonzero = delta[delta != 0]
    p_str = f"{wilcoxon(nonzero).pvalue:.2e}" if len(nonzero) > 10 else "N/A"
    print(f"  {name:<20} {delta.mean():>+10.4f} {np.median(delta):>+10.4f} {pos_pct:>7.1f}% {p_str:>12}")

print(f"\n📊 F1 상세")
print(f"  {'':>15} {'orig':>10} {'swap':>10}")
print(f"  {'mean':>15} {orig_f1.mean():>10.3f} {swap_f1.mean():>10.3f}")
print(f"  {'median':>15} {np.median(orig_f1):>10.3f} {np.median(swap_f1):>10.3f}")

print(f"\n📊 IoU 상세")
print(f"  {'':>15} {'orig':>10} {'swap':>10}")
print(f"  {'mean':>15} {orig_iou.mean():>10.3f} {swap_iou.mean():>10.3f}")
print(f"  {'median':>15} {np.median(orig_iou):>10.3f} {np.median(swap_iou):>10.3f}")

print(f"\n📊 Density bucket별 Δ F1")
print(f"  {'bucket':<10} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")
for b in range(N_BUCKETS):
    mask = buckets_d == b
    if mask.sum() == 0: continue
    d = delta_f1[mask]
    nonzero = d[d != 0]
    p_str = f"{wilcoxon(nonzero).pvalue:.2e}" if len(nonzero) > 10 else "N/A"
    print(f"  {b:<10} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}% {p_str:>12}")

print(f"\n📊 인스턴스 수 구간별 Δ F1")
print(f"  {'구간':<15} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
for i, (lo, hi) in enumerate([(0,1),(1,10),(10,100),(100,1000),(1000,10000)]):
    mask = (gt_ns_d >= lo) & (gt_ns_d < hi)
    if mask.sum() == 0: continue
    d = delta_f1[mask]
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}%")

print(f"\n📊 Density bucket별 Δ IoU")
print(f"  {'bucket':<10} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
for b in range(N_BUCKETS):
    mask = buckets_d == b
    if mask.sum() == 0: continue
    d = delta_iou[mask]
    print(f"  {b:<10} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}%")

📊 상대적 차이 (Δ = orig - swap, n=3,320)

  메트릭                      Δ mean   Δ median    Δ>0 %   Wilcoxon p
  ──────────────────── ────────── ────────── ──────── ────────────
  F1                      +0.2161    +0.1343    63.6%    1.24e-308
  IoU                     +0.1871    +0.1102    63.5%     0.00e+00
  Coverage (|1-x|)     +26481265065.9615    +0.0471    55.1%    3.83e-112

📊 F1 상세
                        orig       swap
             mean      0.391      0.175
           median      0.442      0.029

📊 IoU 상세
                        orig       swap
             mean      0.302      0.114
           median      0.284      0.015

📊 Density bucket별 Δ F1
  bucket          n     Δ mean   Δ median    Δ>0 %   Wilcoxon p
  ────────── ────── ────────── ────────── ──────── ────────────
  0             665    +0.0428    +0.0000    13.7%     6.72e-10
  1             665    +0.1758    +0.0010    51.1%     1.62e-43
  2             660    +0.2858    +0.2634    76.7%     1.17e-77
  3             66